# Perturbation Analysis

Single-step in-silico TF perturbation screening via the ReprogrammingPipeline.
Covers: perturbation results table, candidate priority/summary table, hit classification.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.utils.config import load_yaml_config
from model.utils.reproducibility import seed_everything
from model.utils.device import get_device
from model.utils.constants import (
    get_pv_path_nodes,
    get_cluster_column,
    get_time_column,
    get_spotlight_tfs,
    get_main_metrics,
    get_diagnostic_metrics,
)
from model.models import Classifier, ClassifierConfig, Forecaster, ForecasterConfig
from model.training.checkpointing import load_checkpoint
from model.data.preprocessing import prepare_classifier_data, prepare_clusters
from model.analysis import (
    CellDataProcessor,
    FateClassifierEvaluator,
    PathSpecificReferenceProjector,
    PerturbationRunner,
    ProcessorConfig,
    ReprogrammingPipeline,
    aggregate_repeated_experiment_summary,
    build_rule_based_hit_classes,
    split_panel_summary_tables,
    compute_priority_scores,
    canonical_metric_name,
    is_control_label,
    compute_multi_readout_scores,
    classify_response_modes,
    build_candidate_random_calibration_df,
)
from model.analysis.stats import bootstrap_metric

DEVICE = get_device()
print('DEVICE:', DEVICE)


In [ ]:
# ===== Parameters =====
SMOKE_MODE = True            # False -> fuller perturbation screen
SEED = 42
seed_everything(SEED)

N_CELLS = 50 if SMOKE_MODE else 200          # cells sampled per repeat
N_REPEATS = 2 if SMOKE_MODE else 5           # repeat seeds per perturbation
N_RANDOM_CTRL = 3 if SMOKE_MODE else 10      # random-TF controls
N_TF_SPOTLIGHT = 5 if SMOKE_MODE else 10     # top N spotlight TFs to screen
INFER_BATCH_SIZE = 32 if SMOKE_MODE else 64
TARGET_TIME_HORIZON = 0.1

RUN_CLASSIFIER_READOUT = True   # requires multi-class classifier

OUTDIR = ROOT / 'artifacts' / 'notebooks' / 'perturbation'
OUTDIR.mkdir(parents=True, exist_ok=True)

# Configs
CLS_MULTI_CFG = load_yaml_config(ROOT / 'configs' / 'classifier_multi.yaml')
GEN_CFG = load_yaml_config(ROOT / 'configs' / 'forecasting.yaml')

adata_path = ROOT / GEN_CFG['data']['h5ad_path']
cls_multi_ckpt = ROOT / CLS_MULTI_CFG['training']['checkpoint_path']
gen_ckpt_path = ROOT / GEN_CFG['training']['checkpoint_path']

print('adata_path:', adata_path)
print('cls_multi_ckpt:', cls_multi_ckpt)
print('gen_ckpt_path:', gen_ckpt_path)
print('SMOKE_MODE:', SMOKE_MODE)

In [ ]:
# Load data and models
adata_full = sc.read_h5ad(adata_path)
adata_full = prepare_clusters(adata_full, get_cluster_column())
print(adata_full)

# Rebuild classifier preprocessing metadata so readouts use the real class-index order.
_, cls_multi_stats = prepare_classifier_data(
    h5ad_path=str(adata_path),
    max_len=int(CLS_MULTI_CFG['data'].get('max_len', 2000)),
    exclude_class=CLS_MULTI_CFG['data'].get('exclude_class'),
    label_mode='multi',
    cluster_col=CLS_MULTI_CFG['data'].get('cluster_col'),
    verbose=False,
)
print('Multi-class class_names:', cls_multi_stats['class_names'])

# Evaluation-only held-out cells: prefer explicit split annotations, else stratify by cluster.
def build_heldout_adata(adata_obj, cluster_col, heldout_split=0.1, seed=42):
    heldout_tokens = {'heldout', 'holdout', 'test', 'eval', 'evaluation'}
    for col in ['split', 'data_split', 'dataset_split', 'set']:
        if col in adata_obj.obs.columns:
            split_vals = adata_obj.obs[col].astype(str).str.lower().str.strip()
            mask = split_vals.isin(heldout_tokens).to_numpy()
            if mask.any():
                return adata_obj[mask].copy(), f'obs[{col}]'

    if cluster_col not in adata_obj.obs.columns:
        raise ValueError(f'cluster_col {cluster_col!r} not found in adata.obs')

    rng = np.random.default_rng(seed)
    heldout_idx = []
    cluster_vals = adata_obj.obs[cluster_col].astype(str).to_numpy()
    for cl in np.unique(cluster_vals):
        idx = np.where(cluster_vals == cl)[0]
        if len(idx) == 0:
            continue
        n_hold = int(round(len(idx) * heldout_split))
        n_hold = min(max(n_hold, 1), len(idx))
        heldout_idx.append(rng.choice(idx, size=n_hold, replace=False))
    if not heldout_idx:
        raise ValueError('Failed to build held-out subset: no eligible cells')
    heldout_idx = np.sort(np.concatenate(heldout_idx))
    return adata_obj[heldout_idx].copy(), 'stratified_by_cluster'

adata_perturb, heldout_source = build_heldout_adata(
    adata_obj=adata_full,
    cluster_col=get_cluster_column(),
    heldout_split=0.1,
    seed=SEED,
)
print(f'Full cells: {adata_full.n_obs} | Held-out cells: {adata_perturb.n_obs} ({heldout_source})')

# Backward-compatible handle for later cells that only need gene names.
adata = adata_full

# Multi-class classifier (for fate readout)
cls_multi_cfg_dict = dict(CLS_MULTI_CFG['model'])
if cls_multi_cfg_dict.get('vocab_size') in [None, 'null']:
    cls_multi_cfg_dict['vocab_size'] = int(adata_full.n_vars) + 1
if cls_multi_cfg_dict.get('num_classes') in [None, 'null']:
    cls_multi_cfg_dict['num_classes'] = int(cls_multi_stats['num_classes'])

multi_classifier = Classifier(ClassifierConfig.from_dict(cls_multi_cfg_dict)).to(DEVICE)
multi_classifier, _ = load_checkpoint(multi_classifier, cls_multi_ckpt, device=DEVICE)
multi_classifier.eval()
print('Multi-class classifier loaded.')

# Forecaster
f_model = Forecaster(ForecasterConfig.from_dict(GEN_CFG['model'])).to(DEVICE)
f_model, _ = load_checkpoint(f_model, gen_ckpt_path, device=DEVICE)
f_model.eval()
print('Forecaster loaded.')


In [ ]:
# Build pipeline components
proc_cfg = ProcessorConfig(
    n_genes=int(GEN_CFG['model']['n_genes']),
    gen_max_len=int(GEN_CFG['model'].get('max_len', 1000)),
    cls_max_len=int(CLS_MULTI_CFG['data'].get('max_len', 2000)),
    cluster_col=get_cluster_column(),
    time_col=get_time_column(),
    class_names=tuple(cls_multi_stats['class_names']),
)

# Perturbation is evaluated on held-out cells; the reference geometry is fitted on the full PV path.
processor = CellDataProcessor(
    adata=adata_perturb,
    generator=f_model,
    classifier=multi_classifier,
    config=proc_cfg,
    device=DEVICE,
)

perturb_runner = PerturbationRunner(
    processor=processor,
    infer_batch_size=INFER_BATCH_SIZE,
    target_time_horizon=TARGET_TIME_HORIZON,
)

projector = PathSpecificReferenceProjector(
    adata=adata_full,
    n_genes=proc_cfg.n_genes,
    path_clusters=get_pv_path_nodes(),
    cluster_col=proc_cfg.cluster_col,
    time_col=proc_cfg.time_col,
    n_pca_components=20,
)

classifier_eval = FateClassifierEvaluator(
    processor=processor,
    terminal_cluster_label='15',
    path_cluster_sequence=get_pv_path_nodes(),
) if RUN_CLASSIFIER_READOUT else None

pipeline = ReprogrammingPipeline(
    processor=processor,
    perturb_runner=perturb_runner,
    projector=projector,
    classifier_evaluator=classifier_eval,
)
print('Pipeline ready.')
print(f'Perturbation cells: {processor.adata.n_obs} held-out cells ({heldout_source})')
print(f'Reference progress sign: {projector.progress_sign:+.0f}')
if classifier_eval is not None:
    print(f'Terminal cluster 15 class index: {classifier_eval.terminal_cluster_idx} / class_names={classifier_eval.class_names}')


In [ ]:
# Define perturbation specs
spotlight_tfs = get_spotlight_tfs()[:N_TF_SPOTLIGHT]
print('Spotlight TFs:', spotlight_tfs)

perturbation_specs = {}

# OE and KD for each spotlight TF
for tf in spotlight_tfs:
    if processor.has_gene(tf):
        perturbation_specs[f'{tf}_OE'] = {tf: 'OE'}
        perturbation_specs[f'{tf}_KD'] = {tf: 'KD'}
    else:
        print(f'  [WARN] Gene {tf} not found in adata, skipping')

# Random-TF controls
all_gene_names = list(adata.var_names[:proc_cfg.n_genes])
rng_ctrl = np.random.default_rng(SEED)
for i in range(N_RANDOM_CTRL):
    random_tf = str(rng_ctrl.choice(all_gene_names, size=1)[0])
    perturbation_specs[f'RANDOM_CTRL_{i}_{random_tf}'] = {random_tf: 'OE'}

print(f'\nTotal perturbation specs: {len(perturbation_specs)}')
for label in sorted(perturbation_specs.keys()):
    print(f'  {label}: {perturbation_specs[label]}')

In [ ]:
# Run single-step perturbations across repeats
per_cell_records = []  # list of (label, seed, metric_dict_per_cell)
single_step_table_rows = []

main_metrics = get_main_metrics()
diag_metrics = get_diagnostic_metrics()
all_tracked_metrics = list(main_metrics) + [m for m in diag_metrics if m not in main_metrics]

for label, spec in perturbation_specs.items():
    print(f'\n--- {label} ---')
    for repeat_seed in range(N_REPEATS):
        cell_seed = SEED * 1000 + repeat_seed
        try:
            sampled = processor.sample_cells(
                index_label='all', n_samples=N_CELLS, seed=cell_seed
            )
        except Exception as e:
            print(f'  [SKIP] seed={repeat_seed}: sample_cells error: {e}')
            continue

        try:
            result = pipeline.compare_generator_experiment(
                sampled_cells=sampled,
                perturbation_spec=spec,
                run_classifier=RUN_CLASSIFIER_READOUT,
            )
        except Exception as e:
            print(f'  [SKIP] seed={repeat_seed}: pipeline error: {e}')
            continue

        geom = result['geometry_comparison']
        n_cells = len(geom.get('delta_path_progress', []))

        # Compute delta_path_deviation (orthogonal to PC1)
        pca_disp = np.asarray(geom['pca_displacement'], dtype=np.float64)
        path_prog = np.asarray(geom['delta_path_progress'], dtype=np.float64)
        delta_path_deviation = np.sqrt(np.maximum(0, pca_disp**2 - path_prog**2))

        for i in range(n_cells):
            row = {
                'label': label,
                'repeat_seed': repeat_seed,
                'cell_row': i,
                'delta_path_progress': float(path_prog[i]),
                'delta_path_deviation': float(delta_path_deviation[i]),
            }
            if RUN_CLASSIFIER_READOUT and 'classifier_comparison' in result:
                cls_comp = result['classifier_comparison']
                row['delta_logit_cluster15'] = float(cls_comp['delta_logit_cluster15'][i])
                row['delta_path_index_expectation'] = float(cls_comp['delta_path_index_expectation'][i])
            per_cell_records.append(row)

        # Build single-step summary row for this repeat
        is_ctrl = is_control_label(label)
        summary_row = {
            'label': label,
            'repeat_seed': repeat_seed,
            'n_cells': n_cells,
            'is_control': is_ctrl,
            'delta_path_progress_mean': float(np.mean(path_prog)),
            'delta_path_deviation_mean': float(np.mean(delta_path_deviation)),
        }
        if RUN_CLASSIFIER_READOUT and 'classifier_comparison' in result:
            cls_comp = result['classifier_comparison']
            summary_row['delta_logit_cluster15_mean'] = float(np.mean(cls_comp['delta_logit_cluster15']))
            summary_row['delta_path_index_expectation_mean'] = float(np.mean(cls_comp['delta_path_index_expectation']))
        single_step_table_rows.append(summary_row)
        print(f'  seed={repeat_seed}: n_cells={n_cells}, path_prog_mean={summary_row["delta_path_progress_mean"]:.4f}')

per_cell_df = pd.DataFrame(per_cell_records)
single_step_df = pd.DataFrame(single_step_table_rows)
print(f'\nTotal per-cell records: {len(per_cell_df)}')
print(f'Single-step summary rows: {len(single_step_df)}')

In [ ]:
# Build per-cell dict for aggregate_repeated_experiment_summary
per_cell_dfs = {}
for label, grp in per_cell_df.groupby('label', observed=True):
    if grp['repeat_seed'].nunique() >= 2:
        per_cell_dfs[label] = grp.copy()

if per_cell_dfs:
    panel_summary_df = aggregate_repeated_experiment_summary(
        per_cell_dfs=per_cell_dfs,
        metrics=None,  # default: MAIN_METRICS + DIAGNOSTIC_METRICS
    )
    print(f'Panel summary: {len(panel_summary_df)} rows')
    display(panel_summary_df.head(20))
else:
    panel_summary_df = pd.DataFrame()
    print('Not enough repeats for panel summary.')

In [ ]:
# Hit classification
if not panel_summary_df.empty:
    hit_df = build_rule_based_hit_classes(
        panel_summary_df=panel_summary_df,
        fdr_threshold=0.05,
    )
    print('Hit classification:')
    display(hit_df)
    print(f'\nHit class counts:\n{hit_df["hit_class"].value_counts().to_string()}')
else:
    hit_df = pd.DataFrame(columns=['label', 'hit_class'])
    print('Skipped hit classification: empty panel summary.')

## Bootstrap CI on Readout Metrics

In [ ]:
# Bootstrap CI for each label x metric using per-cell delta values
print('Computing bootstrap CIs ...')
n_boot = 200 if SMOKE_MODE else 1000

# Melt per_cell_df to long format for bootstrapping
value_cols = [c for c in per_cell_df.columns if c.startswith('delta_')]
per_cell_long = per_cell_df.melt(
    id_vars=['label', 'repeat_seed', 'cell_row'],
    value_vars=value_cols,
    var_name='metric',
    value_name='delta',
)

bootstrap_rows = []
for (label, metric), grp in per_cell_long.groupby(['label', 'metric'], observed=True):
    values = grp['delta'].dropna().to_numpy()
    if len(values) < 5:
        continue
    try:
        result = bootstrap_metric(values, stat='mean', n_boot=n_boot, ci=0.95, seed=SEED)
        bootstrap_rows.append({
            'label': label,
            'metric': metric,
            'point_estimate': result['point_estimate'],
            'ci_low': result['ci_low'],
            'ci_high': result['ci_high'],
            'boot_mean': result['boot_mean'],
            'boot_std': result['boot_std'],
        })
    except Exception as e:
        print(f'  [SKIP] bootstrap failed for {label}/{metric}: {e}')

bootstrap_df = pd.DataFrame(bootstrap_rows)
if not bootstrap_df.empty:
    print(f'Bootstrap CI computed for {len(bootstrap_df)} label x metric combinations')
    display(bootstrap_df.head(20))
else:
    print('[WARN] No bootstrap results produced')


## Multi-Readout Scoring & Response Mode Classification

In [ ]:
# Split panel summary into main and diagnostic metric tables
if not panel_summary_df.empty:
    panel_main, panel_diag = split_panel_summary_tables(panel_summary_df)
else:
    panel_main = pd.DataFrame()
    panel_diag = pd.DataFrame()

# Pivot panel_main from long to wide for multi-readout scoring
if not panel_main.empty:
    panel_wide = panel_main.pivot_table(
        index='label', columns='metric', values='repeat_mean_of_means'
    ).reset_index()
    panel_wide['is_control'] = panel_wide['label'].apply(is_control_label)

    # Compute oriented z-scores for each readout metric
    scores_df = compute_multi_readout_scores(panel_wide)
    print(f'Multi-readout scores: {len(scores_df)} candidates')
    display(scores_df[['label', 'is_control', 'multi_readout_score', 'dominant_readout']].head(20))

    # Classify response modes (concordant / identity-biased / geometry-biased / mixed-weak)
    try:
        response_df = classify_response_modes(scores_df, z_threshold=0.5)
        print('\nResponse mode distribution:')
        print(response_df['response_mode'].value_counts().to_string())
        display(response_df[['label', 'response_mode', 'multi_readout_score']].head(20))
    except Exception as e:
        print(f'[WARN] Response mode classification failed: {e}')
        response_df = pd.DataFrame()
else:
    scores_df = pd.DataFrame()
    response_df = pd.DataFrame()
    print('[WARN] panel_main is empty; skipping multi-readout scoring')


## Candidate vs Random Calibration

In [ ]:
# Save single-step results
single_step_df.to_csv(OUTDIR / 'single_step_results.csv', index=False)
print('Saved:', OUTDIR / 'single_step_results.csv')

# Save panel summary tables (already split in multi-readout section)
if not panel_main.empty:
    panel_main.to_csv(OUTDIR / 'panel_summary_main.csv', index=False)
    print('Saved:', OUTDIR / 'panel_summary_main.csv')

if not panel_diag.empty:
    panel_diag.to_csv(OUTDIR / 'panel_summary_diagnostic.csv', index=False)
    print('Saved:', OUTDIR / 'panel_summary_diagnostic.csv')

if not panel_summary_df.empty:
    panel_summary_df.to_csv(OUTDIR / 'panel_summary_full.csv', index=False)
    print('Saved:', OUTDIR / 'panel_summary_full.csv')

# Save hit classification
if not hit_df.empty:
    hit_df.to_csv(OUTDIR / 'hit_classification.csv', index=False)
    print('Saved:', OUTDIR / 'hit_classification.csv')

# Save per-cell raw data
per_cell_df.to_csv(OUTDIR / 'per_cell_delta.csv', index=False)
print('Saved:', OUTDIR / 'per_cell_delta.csv')

# Save bootstrap CI results
if not bootstrap_df.empty:
    bootstrap_df.to_csv(OUTDIR / 'bootstrap_ci.csv', index=False)
    print('Saved:', OUTDIR / 'bootstrap_ci.csv')

# Save multi-readout scores and response modes
if not scores_df.empty:
    scores_df.to_csv(OUTDIR / 'multi_readout_scores.csv', index=False)
    print('Saved:', OUTDIR / 'multi_readout_scores.csv')

if not response_df.empty:
    response_df.to_csv(OUTDIR / 'response_mode_classification.csv', index=False)
    print('Saved:', OUTDIR / 'response_mode_classification.csv')

# Save calibration table
if not calib_df.empty:
    calib_df.to_csv(OUTDIR / 'candidate_vs_random_calibration.csv', index=False)
    print('Saved:', OUTDIR / 'candidate_vs_random_calibration.csv')

print(f'\n=== All perturbation outputs saved to {OUTDIR} ===')
